# Lesson 07 — Contextual retrieval

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/07-contextual-retrieval/colab.ipynb)

*A chunk that says "revenue grew 3%" never says whose revenue. Put that back before embedding.*

**What you will learn**

- Why chunks lose the context that made them retrievable
- Anthropic's contextual-retrieval technique: an LLM writes a situating prefix per chunk
- Why the prefix is stored *separately* from the chunk text (citations!)
- Why the keyword index must see the prefix too — and how to prove it
- How prompt caching, the Batch API and a cheap model make it affordable — with the token arithmetic worked out

| | |
|---|---|
| ⏱️ Time | 45–60 min |
| 🖥️ Runtime | **T4 GPU strongly recommended** — one model call per chunk (~113 calls) |
| 🤖 Needs the generator (Ollama) | yes — for some cells |

**Before this:** [06-query-transformation](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/06-query-transformation/colab.ipynb)

> **How to use this notebook.** Run the cells top to bottom (`Shift+Enter`). Every
> time you see **🔮 Predict**, stop and write down what you expect *before* running
> the next cell — the gap between prediction and result is where the learning is.
> Answers to the check-your-understanding questions are hidden behind ▶ toggles.

## 0. Setup

This cell clones the course repo into the Colab runtime, installs the
dependencies, and (if `USE_LLM` is ticked) installs and starts
[Ollama](https://ollama.com) with the small `qwen2.5:1.5b` generator. It is safe
to re-run. Everything in the repo's `ragkit/` library becomes importable.

If an import fails after the install, use **Runtime → Restart session** and run
this cell again.

In [ ]:
# @title ⚙️ Setup — run me first
USE_LLM = True  # @param {type:"boolean"}
BRANCH = "main"  # @param {type:"string"}

import subprocess
import sys
from pathlib import Path


def _find_repo():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "ragkit").is_dir() and (p / "colab" / "colab_setup.py").exists():
            return p
    return None


REPO = _find_repo()
if REPO is None:
    REPO = Path("/content/rag-from-scratch-to-production")
    if not REPO.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                BRANCH,
                "https://github.com/safar-byte/rag-from-scratch-to-production.git",
                str(REPO),
            ],
            check=True,
        )
sys.path.insert(0, str(REPO / "colab"))
from colab_setup import compare_runs, eval_frames, results_frame, setup  # noqa: F401

LLM_READY = setup(llm=USE_LLM)

## 1. The problem, concretely

A chunk that reads, in full:

> Revenue grew 3% over the previous quarter.

Which company? Which quarter? The chunk does not say, because the *surrounding document*
said it. Embedded as-is, it will never be retrieved by "How did Acme do in Q2 2024?" —
those words appear nowhere in it.

This is not an edge case. Pronouns, relative dates and unqualified references ("the
system", "this approach", "the above") all produce chunks that are individually
unretrievable. The information is in the corpus and unreachable.

### Diagnose before you spend anything

The single best predictor of whether this technique will pay on *your* corpus: read
some random chunks **with the document hidden**. How many make sense on their own?

In [ ]:
import random

from ragkit.config import get_settings
from ragkit.ingest import RecursiveChunker, chunk_documents, load_documents

settings = get_settings()
documents = load_documents()
chunks = chunk_documents(documents, RecursiveChunker(settings.chunk_size, settings.chunk_overlap))

random.seed(7)
for c in random.sample([c for c in chunks if c.start_char > 0], 4):
    print("─" * 80 + f"\n{c.text[:350]}\n")

For each one, ask: could a user's question find this? Would *you* know what "it" or
"this" refers to? On this corpus most chunks are fairly self-contained (the documents
are short explanatory essays with headings), so expect a modest gain. A corpus of
meeting notes, threads or narrative reports would gain far more.

## 2. The fix

Before embedding, ask a cheap model to write one or two sentences situating the chunk in
its document, and prepend that:

> *This chunk is from Acme Corp's Q2 2024 earnings report, in the section on the cloud
> infrastructure segment.*
>
> Revenue grew 3% over the previous quarter.

Now "Acme" and "Q2 2024" are in the embedded text. The prompt `ragkit` uses:

In [ ]:
from ragkit.ingest.contextual import CONTEXT_SYSTEM

print(CONTEXT_SYSTEM)

Note "resolve pronouns and vague references" and "do not add information the document
does not contain" — the prefix should *locate* the chunk, not summarise or embellish it.

## 3. Preview a few prefixes

In [ ]:
from pathlib import Path

from ragkit.ingest.contextual import ContextualiseCache, contextualise
from ragkit.providers import get_generator

cache = ContextualiseCache(Path(".cache/contextual.json"))  # keyed on document content
print(f"{len(cache)} prefixes already cached")

if LLM_READY:
    generator = get_generator(settings, cheap=True)  # the cheap model, on purpose
    preview = [c for c in chunks if c.start_char > 0][:4]  # mid-document chunks lose the most
    contextualise(preview, documents, generator, cache=cache, progress=False)
    for c in preview:
        print("─" * 80)
        print("CONTEXT :", c.context)
        print("TEXT    :", c.text[:220].replace("\n", " "), "…\n")

## 4. Two details that are easy to get wrong

### The prefix is stored separately from the text

`Chunk.context` and `Chunk.text` stay distinct; `Chunk.embed_text` joins them for
embedding. If you concatenate at ingest instead, every later citation points at
*generated* text rather than source text — and the system can no longer prove anything
it says. (That is the question lesson 00 asked you to guess.)

In [ ]:
if LLM_READY:
    c = preview[0]
    print("what gets EMBEDDED (embed_text):\n", c.embed_text[:300], "…\n")
    print("what gets SHOWN and CITED (text):\n", c.text[:200], "…")

### The lexical index must get the prefix too

The prefix adds exactly the keywords a user is likely to type. `Bm25Index` indexes
`embed_text`, not `text`, for this reason. Indexing the prefix for dense search only
leaves half the value unclaimed — and nothing will tell you that you did. Section 6
proves it.

## 5. Contextualise the whole corpus and measure

~113 model calls. On a T4 this is a couple of minutes; on CPU it can be 10–45 minutes
(the author's machine took ~25s per chunk). Results are **cached on disk, keyed on the
document's content hash**, so a re-run is free — and editing a document invalidates
exactly the prefixes derived from it. A stale prefix describing an older version of a
document is worse than no prefix: it is confidently wrong.

We build the contextual index under `/tmp` so the course's main index (used by every
other lesson) is untouched.

In [ ]:
import time

from ragkit.config import Settings
from ragkit.eval.run import run_eval
from ragkit.pipeline import RagPipeline

BUILD = True  # @param {type:"boolean"}

plain = RagPipeline(settings, strategy="rerank")
if plain.count() == 0:
    print(plain.ingest())

contextual = None
if LLM_READY and BUILD:
    started = time.perf_counter()
    contextualise(chunks, documents, generator, cache=cache)
    print(f"contextualised {len(chunks)} chunks in {time.perf_counter() - started:.0f}s")

    ctx_settings = Settings(chroma_path=Path("/tmp/l07/.chroma"))
    contextual = RagPipeline(ctx_settings, strategy="rerank")
    contextual._embedder, contextual._reranker = plain.embedder, plain.reranker  # reuse models
    contextual._store.reset()
    contextual._store.add(
        chunks, contextual.embedder.embed_documents([c.embed_text for c in chunks])
    )
    contextual._bm25.build(chunks)  # BM25 over embed_text: prefix included
    print("contextual index:", contextual.count(), "chunks")

In [ ]:
runs = {}
for strategy in ["dense", "rerank"]:
    plain.set_strategy(strategy)
    runs[f"{strategy}"] = run_eval(pipeline=plain, retrieval_only=True, quiet=True)
    if contextual is not None:
        contextual.set_strategy(strategy)
        runs[f"{strategy} + contextual"] = run_eval(
            pipeline=contextual, retrieval_only=True, quiet=True
        )
display(compare_runs(runs).round(3))
print("vocab_mismatch:")
display(compare_runs(runs, kind="vocab_mismatch").round(3))

🔮 **Before reading on:** did it help? Where? Did it help *both* rows equally?

When this notebook was written (qwen2.5:1.5b writing the prefixes), the result split in
two:

| | R@3 | R@5 | vocab R@5 |
|---|---|---|---|
| dense → dense + contextual | 0.943 → **0.977** | 0.943 → **0.977** | 0.750 → **1.000** |
| rerank → rerank + contextual | 0.966 → 0.920 | 0.989 → 0.966 | 1.000 → 0.875 |

**Contextual retrieval clearly helped the embedder — and made the reranked pipeline
worse.** Contextual retrieval had never been measured in this repo before this notebook
was written, so this is a fresh finding. Let's see which questions moved under dense:

In [ ]:
import pandas as pd

from ragkit.eval.golden import load_golden


def first_rank(pipe, q):
    docs = list(dict.fromkeys(s.chunk.doc_id for s in pipe.retrieve(q.question, top_k=20)))
    return next((i for i, d in enumerate(docs, 1) if d in q.relevant), None)


if contextual is not None:
    plain.set_strategy("dense")
    contextual.set_strategy("dense")
    moved = []
    for q in [q for q in load_golden() if q.relevant]:
        before, after = first_rank(plain, q), first_rank(contextual, q)
        if before != after:
            moved.append(
                {
                    "question": q.question[:70],
                    "kind": q.kind,
                    "plain rank": before,
                    "contextual rank": after,
                }
            )
    display(pd.DataFrame(moved))

### Why would the reranker get worse?

Look back at the prefixes from section 3 and read them critically:

- They begin *"The excerpt discusses…"* — which the prompt explicitly forbids. A cheap
  1.5B model drifts from instructions.
- Some **add facts that are not in the document** (e.g. listing embedding sizes the
  source never mentions) — the prompt forbids that too.

Now note *what the reranker reads*: `CrossEncoderReranker` scores `chunk.embed_text`,
i.e. **prefix + text**. An embedding averages a noisy prefix into one vector; a
cross-encoder reads every word and attends to the prefix's boilerplate and invented
details. Hypothesis: **the prefix helps the bi-encoder find candidates, but misleads the
cross-encoder that orders them.**

A hypothesis is only worth something if you test it. Rerank the *same* contextual
shortlist, but let the cross-encoder read only the original `text`:

In [ ]:
import copy

from ragkit.retrieve import Bm25Retriever, DenseRetriever, HybridRetriever
from ragkit.retrieve.rerank import RerankingRetriever


class RerankOnSourceText:
    """Wraps a reranker so it scores chunk.text, not the generated prefix."""

    def __init__(self, inner):
        self.inner, self.name = inner, f"{inner.name}(text-only)"

    def rerank(self, query, chunks, top_k):
        stripped = {id(c): c for c in chunks}
        bare = []
        for c in chunks:
            b = copy.copy(c)
            b.context = ""  # embed_text falls back to text
            stripped[id(b)] = c
            bare.append(b)
        return [(stripped[id(b)], s) for b, s in self.inner.rerank(query, bare, top_k)]


if contextual is not None:
    shortlist = HybridRetriever(
        [DenseRetriever(contextual._store, contextual.embedder), Bm25Retriever(contextual._bm25)],
        candidates=10,
    )
    contextual._retriever = RerankingRetriever(
        shortlist, RerankOnSourceText(contextual.reranker), candidates=10
    )
    runs["rerank + contextual (CE reads text only)"] = run_eval(
        pipeline=contextual, retrieval_only=True, quiet=True
    )
    contextual.set_strategy("rerank")
    display(compare_runs(runs).round(3))

When this notebook was written, the text-only reranker recovered **everything** the
prefix had cost — back to R@5 0.989 and vocab R@5 1.000, level with plain reranking.
So the hypothesis held: **use the prefix to *find* candidates, not to *judge* them.**

Two honest caveats:

1. Contextual + text-only reranking only *ties* plain reranking here. On this corpus,
   hybrid + rerank had already solved the vocabulary-mismatch questions (lesson 05), so
   there was nothing left for contextual retrieval to add on top. Where it shines is
   the cheaper dense-only pipeline.
2. A stronger prefix writer (Claude Haiku on the cloud profile) follows the prompt and
   does not invent facts. The reranker result may well flip with better prefixes —
   which is precisely why it should be *measured*, not assumed.

## 6. Prove the lexical half matters

Build a second BM25 index over the **same contextualised chunks** but index only
`text` (no prefix), and compare BM25 alone.

In [ ]:
from ragkit.retrieve import Bm25Index, Bm25Retriever

if contextual is not None:
    no_prefix = [copy.copy(c) for c in chunks]
    for c in no_prefix:
        c.context = ""  # embed_text falls back to text
    text_only = Bm25Index(Path("/tmp/l07/.bm25-text-only"))
    text_only.build(no_prefix)

    bm25_runs = {}
    contextual._retriever = Bm25Retriever(text_only)
    bm25_runs["BM25 over text only"] = run_eval(
        pipeline=contextual, retrieval_only=True, quiet=True
    )
    contextual.set_strategy("bm25")
    bm25_runs["BM25 over context + text"] = run_eval(
        pipeline=contextual, retrieval_only=True, quiet=True
    )
    display(compare_runs(bm25_runs).round(3))

Whatever difference you see is value that a "dense-only" implementation of contextual
retrieval silently throws away.

## 7. Making it affordable (the cloud path)

Offline, none of this costs money — it is just slow. On a paid API, the naive
implementation sends **the whole document once per chunk**: a 200-chunk document is
transmitted 200 times. Three mechanisms fix it, in descending order of effect:

1. **Prompt caching.** The document is a stable prefix across every chunk derived from
   it, so it is processed once and then read from cache at ~10% of the input price.
   `ClaudeGenerator.generate(cache_system=True)` makes it a cache breakpoint. Verify with
   `usage.cache_read_input_tokens > 0` — a zero means something in the prefix varies
   between requests and you are paying full price silently.
2. **The Batch API**, ~50% off. A one-time ingest pass has no latency requirement —
   exactly what batch is for.
3. **A cheap model.** Situating a chunk is an easy task: `get_generator(cheap=True)`.

Let's do the arithmetic for this corpus with real token counts. (The prices are the
ones `ragkit/providers/cloud.py` uses for the cost column; check current pricing before
relying on them.)

In [ ]:
import tiktoken

from ragkit.providers.cloud import CACHE_READ_DISCOUNT, PRICES_USD_PER_MTOK

enc = tiktoken.get_encoding("cl100k_base")  # a close-enough proxy for Claude's tokenizer
in_price, out_price = PRICES_USD_PER_MTOK["claude-haiku-4-5"]
doc_tokens = {d.doc_id: len(enc.encode(d.text)) for d in documents}
chunk_tokens = [len(enc.encode(c.text)) + 60 for c in chunks]  # + instructions
OUT = 50  # tokens per prefix

naive_in = sum(doc_tokens[c.doc_id] + t for c, t in zip(chunks, chunk_tokens, strict=True))
first_seen, cached_write, cached_read = set(), 0, 0
for c in chunks:
    if c.doc_id in first_seen:
        cached_read += doc_tokens[c.doc_id]
    else:
        first_seen.add(c.doc_id)
        cached_write += doc_tokens[c.doc_id]
fresh = sum(chunk_tokens)
out_cost = len(chunks) * OUT * out_price

costs = {
    "naive (document re-sent per chunk)": naive_in * in_price + out_cost,
    "+ prompt caching": (
        fresh * in_price
        + cached_write * in_price * 1.25
        + cached_read * in_price * CACHE_READ_DISCOUNT
        + out_cost
    ),
}
costs["+ caching + Batch API (~50%)"] = costs["+ prompt caching"] * 0.5
pd.DataFrame(
    {
        "USD for this corpus": {k: v / 1e6 for k, v in costs.items()},
        "× 1,000 corpora this size": {k: v / 1e3 for k, v in costs.items()},
    }
).round(4)

Tiny in absolute terms for 16 documents — but look at the **ratio** between the rows.
That ratio is what decides whether contextualising a million-document corpus is a
rounding error or a budget line.

> One caveat on caching: there is a **minimum cacheable prefix length**. A document
> shorter than it is simply not cached, and `cache_read_input_tokens` stays at zero.
> Another reason to always verify rather than assume.

## 8. When is this worth it?

Contextual retrieval pays in proportion to how **context-dependent** your chunks are.
Self-contained reference entries gain little. Narrative documents, meeting notes,
threaded discussions, anything heavy with pronouns and relative references gain a great
deal. Section 1's diagnostic — read twenty random chunks with the document hidden —
predicts your gain better than any benchmark.

## 9. Check your understanding

**1. Why not just concatenate the generated context into the chunk text at ingest?**

<details><summary>▶ Show answer</summary>

Because citations would then point at model-generated text. The system could no
longer show the user the exact source span it relied on. Keeping `context` separate
lets you *search* with it while *showing and citing* only the original `text`.

</details>

**2. The cache key includes the document's content hash. Why?**

<details><summary>▶ Show answer</summary>

So that editing a document invalidates the prefixes derived from it. A prefix
describing an older version of the document is confidently wrong, which is worse
than no prefix.

</details>

**3. Your prompt-caching dashboard shows `cache_read_input_tokens = 0` on every request. Name two likely causes.**

<details><summary>▶ Show answer</summary>

Something in the prefix varies between requests (a timestamp, a request id, the
chunk placed *before* the document, unsorted JSON), or the prefix is shorter than
the model's minimum cacheable length. Caching is an exact prefix match.

</details>

## ✍️ Exercises

1. **Break the citation guarantee.** Change one chunk so `text = context + text`, build
   the prompt with `ragkit.generate.build_prompt`, and look at what a citation would now
   point to.
2. **Price it for real.** Re-run section 7 assuming a 5,000-document corpus where each
   document is 10× longer. Which row changes most?
3. **Try a different prompt.** Edit `CONTEXT_SYSTEM` to ask for *keywords* instead of
   sentences. Re-index and measure. Better or worse?

## Key takeaways

1. Chunks lose the context that made them findable; an LLM-written prefix puts it back.
2. Embed `context + text`, but show and cite only `text`.
3. Index the prefix in BM25 too, or you leave half the value on the table.
4. Caching + Batch + a cheap model turn a prohibitive ingest pass into a modest one — and caching must be verified, not assumed.
5. The gain depends on how context-dependent your chunks are; diagnose that first.

## Next

**[Lesson 08 — Parent-document retrieval and metadata](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/08-parent-doc-and-metadata/colab.ipynb)**